In [ ]:
import xml.etree.ElementTree as et
import pandas as pd
import re
import os
from src.xml_parse import xml_parse_single, xml_parse_baseline

In [ ]:
xml_file = '../data/PMC012xxxxxx/PMC12000100.xml'
xtree = et.parse(xml_file, parser=et.XMLParser(encoding="UTF-8"))
xroot = xtree.getroot()

In [ ]:
for child in xroot: 
    print(child.tag, child.attrib)
    for child2 in child:
        print('    ', child2.tag, child2.attrib)
        for child3 in child2:
            print('        ', child3.tag, child3.attrib)

In [ ]:
xroot.tag

In [ ]:
def xml_get_attr(node, attr_name: str):
    if attr_name in node.attrib.keys():
        return node.attrib[attr_name]
    else:
        return None     # is it better to return None or empty string?

In [ ]:
xml_get_attr(xroot, '{http://www.w3.org/XML/1998/namespace}lang')

In [ ]:
xml_get_attr(xroot, 'article-type')

In [ ]:
def xml_get_text(nodes, joinstr: str = ''):
    if len(nodes) > 0:
        return joinstr.join(nodes[0].itertext()) #.text
    else: 
        return None

In [ ]:
xml_get_text(xroot.findall('./front/journal-meta/journal-title-group/journal-title'))

In [ ]:
# not all articles have pmid, so also collect pmc id
xml_get_text(xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmid"]'))

In [ ]:
xml_get_text(xroot.findall('./front/article-meta/article-id/[@pub-id-type="pmc"]'))

In [ ]:
xml_get_text(xroot.findall('./front/article-meta/title-group/article-title'))

In [ ]:
def get_country(root):

    locations = ['./front/article-meta/aff/country',
                 './front/article-meta/aff',
                 './front/article-meta/contrib-group/contrib/aff/country',
                 './front/article-meta/contrib-group/aff']
    
    i = 0
    country = None

    while country == None:
        if i == len(locations):
            break

        country = xml_get_text(root.findall(locations[i]))
        i += 1
    
    # some queries return a whole adress, in which case we need the last word of the string 
    if not country == None:
        country = country.split()
        if len(country) > 0:        # check that the string is non-empty
            country = country[-1]
        else:
            country = None          # replace empty string with None
        
    
    return country


In [ ]:
xml_get_text(xroot.findall('./front/article-meta/contrib-group/aff')).split()[-1]

In [ ]:
'test'.split()[-1]

In [ ]:
xroot.findall('./front/article-meta/contrib-group/aff')

In [ ]:
get_country(xroot)

In [ ]:
# attributes to add @pub-type="ppub", 
def get_date(root):
    date = xml_get_text(root.findall('./front/article-meta/pub-date/[@pub-type="epub"]'), '-')

    if date == None:
        date = xml_get_text(root.findall('./front/article-meta/pub-date/[@date-type="pub"]'), '-')

    return date

In [ ]:
get_date(xroot)

In [ ]:
# alternative version: get date dict to include tags to make sure day and month aren't confused
nodes = xroot.findall('./front/article-meta/pub-date/[@pub-type="epub"]')
date = {}
if len(nodes) > 0:
    for child in nodes[0]:
        date[child.tag] = child.text
date

In [ ]:
# some abstracts are structured into background/methods/results, 
# the section titles are part of tha abstracts as of now.
# How to remove?

def xml_get_abstr(node):
    abs = ""
    for a in node:
        if 'graphical' in a.attrib.values():
            continue 
        abs += xml_get_text(node)
    
    return abs

In [ ]:
xml_get_abstr(xroot.findall('./front/article-meta/abstract'))

In [ ]:
xml_parse_single(xml_file)

In [ ]:
paper1 = xml_parse_single('../data/PMC012xxxxxx/PMC12187463.xml')
paper2 = xml_parse_single('../data/PMC012xxxxxx/PMC12187462.xml')
paper3 = xml_parse_single('../data/PMC012xxxxxx/PMC12187460.xml')

In [ ]:
p1 = {k:[v] for k,v in paper1.items()}
p2 = {k:[v] for k,v in paper2.items()}
p3 = {k:[v] for k,v in paper3.items()}

In [ ]:
{k:v.append(p3[k][0]) for k,v in p1.items()}

In [ ]:
p1

In [ ]:
pd.DataFrame.from_dict(p1)

In [ ]:
r = re.compile('PMC\d{8}.xml')
if r.match('PMC12187963.xml'):
    print('yeah')


In [ ]:
baseline_df = xml_parse_baseline('../data', '../data/baseline_PMC12.json')

In [ ]:
baseline_df.head()

In [ ]:
baseline_df.to_json('../data/baseline_PMC12.json')

In [ ]:
baseline_df = pd.read_json('../data/baseline_PMC12.json')

In [ ]:
baseline_df.head()